# Set up GPU, CUDA 9.0 --> tf1.9, keras2.1.6

In [1]:
import sys
sys.path.insert(0,'/home/students/acct1002_02/anaconda3/envs/old_tf_gpu/lib/python3.6/site-packages')

In [2]:
from tensorflow.python.util import deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False   #ignore FutureWarning

import tensorflow as tf
tf.__version__

/home/students/acct1002_02/anaconda3/envs/old_tf_gpu/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:523: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/students/acct1002_02/anaconda3/envs/old_tf_gpu/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:524: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/students/acct1002_02/anaconda3/envs/old_tf_gpu/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:525: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int1

'1.9.0'

# Main Body

# 1.import librabies

In [3]:
import random
from tqdm import tqdm
import json

import pandas as pd
import numpy as np
import numpy.random as nr
from numpy.random import seed
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
import cv2
import tensorflow.keras as keras
from tensorflow.keras import backend as K


from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.model_selection import train_test_split


from tensorflow.keras.models import Sequential, Model, load_model
#from tensorflow.keras.applications.vgg16 import preprocess_input
#from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import Conv2D,Dense,Flatten,Dropout,MaxPooling2D, Activation, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing import image
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.callbacks import Callback

In [4]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [5]:
#clear memory in case of OOM
K.clear_session()

# 2. import data and random split

In [6]:
#set dictionary for disease and its index
disease_class = {'Atelectasis': 1,
                 'Cardiomegaly': 2,
                 'Effusion': 3,
                 'Infiltration': 4,
                 'Mass': 5,
                 'Nodule': 6,
                 'Pneumonia': 7,
                 'Pneumothorax': 8,
                 'Consolidation': 9,
                 'Edema': 10,
                 'Emphysema': 11,
                 'Fibrosis': 12,
                 'Pleural_Thickening': 13,
                 'Hernia': 14,
                 'No Finding': 15}

disease_rev = {v: k for k, v in disease_class.items()}

In [7]:
disease_img = {'Atelectasis': [],
                 'Cardiomegaly': [],
                 'Effusion': [],
                 'Infiltration': [],
                 'Mass': [],
                 'Nodule': [],
                 'Pneumonia': [],
                 'Pneumothorax': [],
                 'Consolidation': [],
                 'Edema': [],
                 'Emphysema': [],
                 'Fibrosis': [],
                 'Pleural_Thickening': [],
                 'Hernia': [],
                 'No Finding':[]}

In [8]:
#import labels of the images
data_ref = pd.read_csv("CY1400/Data_Entry.csv")
pd.options.mode.chained_assignment = None        #ignore the SettingWithCopyWarning


for i in tqdm(range(len(data_ref))):
    #print(i)
    if "|" not in data_ref['Finding Labels'][i]:
        disease_img[data_ref['Finding Labels'][i]].append(data_ref['Image Index'][i])

100%|██████████| 112120/112120 [00:01<00:00, 80640.83it/s]


In [9]:
simp_data_ref = data_ref[["Image Index", "Finding Labels"]]
simp_data_ref.set_index("Image Index", inplace = True)

# Training with Noise/GAN augmented images

In [10]:
number = 70000

img_names = []
for dis in disease_img.keys():
    num = round(number/91324*len(disease_img[dis]))
    for i in range(num):
        img_names.append(disease_img[dis][i])

In [11]:
train_image = []
y = np.zeros(shape = (len(img_names), len(disease_class.keys())))

for i in tqdm(range(len(img_names))):

    #img = image.load_img('Noise/N'+img_names[i],target_size=(112,112,3))
    img = image.load_img('GAN/GG'+img_names[i],target_size=(112,112,3))      #GG:cycleGAN; G:DCGAN
    img = image.img_to_array(img)
    train_image.append(img)
    
    for j in range(len(disease_class.keys())):
        if disease_rev[j+1] == simp_data_ref['Finding Labels'][img_names[i]]:
            y[i][j] = 1
            
X = np.array(train_image)

100%|██████████| 70001/70001 [01:22<00:00, 849.43it/s] 


In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state = 42, test_size = 0.15, stratify = y)

# 3. Data preprocessing

In [13]:
X_train = preprocess_input(X_train)
X_val = preprocess_input(X_val)

# 4. Load Model with Fine-Tune and Customise Loss, Callbacks

In [14]:
modelname = 'Noise.hdf5'
#modelname = 'Gan.hdf5'

In [15]:
#customise metrics of auc in callback -> tf1.9 does not have auc as metrics
#auc as metrics calculate auc for each minibatch --> inaccurate
#manually do auc at the end of the whole epoch

#using macro auc (average auc score from all classes) instead of micro auc(total tp over total fn and tp)
#due to class unbanlance

class RocCallback(Callback):
    def __init__(self,x_v, y_v):
        self.x_val = x_v
        self.y_val = y_v
    def on_train_begin(self, logs={}):
        return

    def on_train_end(self, logs={}):
        return

    def on_epoch_begin(self, epoch, logs={}):
        return

    def on_epoch_end(self, epoch, logs={}):
        y_pred_val = self.model.predict(self.x_val, verbose = 1)
        roc_val = roc_auc_score(self.y_val, y_pred_val, average = 'macro')
        print('\rval_roc-auc: %s' % (str(round(roc_val,4))),end=10*' ')
        return

    def on_batch_begin(self, batch, logs={}):
        return

    def on_batch_end(self, batch, logs={}):
        return
    
roc = RocCallback(X_val, y_val)

In [16]:
#custom loss function
#https://www.programmersought.com/article/60001511310/
def focal_loss(alpha = 0.5, beta = 2.0):

    epsilon = 1.e-7
    def loss_fn2(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
        alpha_t = y_true*alpha + (tf.ones_like(y_true)-y_true)*(1-alpha)
        y_t = tf.multiply(y_true, y_pred) + tf.multiply(1-y_true, 1-y_pred)
        ce = -tf.log(y_t)
        weight = tf.pow(tf.subtract(1., y_t), beta)
        fl = tf.multiply(tf.multiply(weight, ce), alpha_t)
        loss = tf.reduce_mean(fl)
        return loss
    
    return loss_fn2

In [17]:
callbacks_list = [
    #roc,
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss', # Use validation loss to monitor the model
        verbose=1,
        patience = 3 # Stop after 3 epochs with no improvements in validation loss
    ),
    
    tf.keras.callbacks.ModelCheckpoint(
        filepath = modelname, # file where the checkpoint is saved
        monitor = 'val_loss', # Don't overwrite the saved model unless val_loss is worse
        verbose=1,
        save_best_only = True # Only save model if it is the best
    )]

In [18]:
#D_model = load_model(modelname, custom_objects={ 'loss_fn': custom_loss()})
#'''
model_json_path = 'D.json' #'VGG16_112.json' #

with open(model_json_path) as infile:
    json_model = json.load(infile)

D_model = keras.models.model_from_json(json.dumps(json_model), custom_objects = {'GlorotUniform': glorot_uniform()})

weights_path = 'D_weights.h5'#'VGG16_112_weights.h5' #
D_model.load_weights(weights_path)

D_model.compile(Adam(lr=0.0001), loss = focal_loss())
#'''

In [19]:

nr.seed(1234)
tf.set_random_seed(4321)
history = D_model.fit(X_train, y_train, batch_size=10, 
                          validation_data=(X_val, y_val),
                          epochs=20, verbose=1, 
                          callbacks = callbacks_list)


Train on 59500 samples, validate on 10501 samples
Epoch 1/20
59500/59500 [==============================] - 170s 3ms/step - loss: 0.0408 - val_loss: 0.0556

Epoch 00001: val_loss improved from inf to 0.05559, saving model to Noise.hdf5
Epoch 2/20
59500/59500 [==============================] - 166s 3ms/step - loss: 0.0351 - val_loss: 0.0495

Epoch 00002: val_loss improved from 0.05559 to 0.04946, saving model to Noise.hdf5
Epoch 3/20
59500/59500 [==============================] - 166s 3ms/step - loss: 0.0337 - val_loss: 0.0469

Epoch 00003: val_loss improved from 0.04946 to 0.04693, saving model to Noise.hdf5
Epoch 4/20
59500/59500 [==============================] - 166s 3ms/step - loss: 0.0330 - val_loss: 0.0457

Epoch 00004: val_loss improved from 0.04693 to 0.04573, saving model to Noise.hdf5
Epoch 5/20
59500/59500 [==============================] - 166s 3ms/step - loss: 0.0326 - val_loss: 0.0449

Epoch 00005: val_loss improved from 0.04573 to 0.04492, saving model to Noise.hdf5
Epoc

# 5.Train With Original images

In [22]:
number = 50000

img_names = []
for dis in disease_img.keys():
    num = round(number/91324*len(disease_img[dis]))
    for i in range(num):
        img_names.append(disease_img[dis][i])

In [23]:
X = []
train_image = []
y = np.zeros(shape = (len(img_names), len(disease_class.keys())))

for i in tqdm(range(len(img_names))):
    
    img = image.load_img('CY1400/images/'+img_names[i],target_size=(112,112,3))
    img = image.img_to_array(img)
    train_image.append(img)
    
    for j in range(len(disease_class.keys())):
        if disease_rev[j+1] == simp_data_ref['Finding Labels'][img_names[i]]:
            y[i][j] = 1
            
X = np.array(train_image)

100%|██████████| 40000/40000 [07:34<00:00, 87.95it/s] 


In [64]:
X_train, X_te, y_train, y_te = train_test_split(X, y, random_state = 42, test_size = 0.45, stratify = y)
X_test, X_val, y_test, y_val = train_test_split(X_te, y_te, random_state = 42, test_size = 0.33, stratify = y_te)


#0.45, 0.33 - when model has been trained with augmentation
#0.3, 0.5 - just original images

In [65]:
X_train = preprocess_input(X_train)
X_test = preprocess_input(X_test)
X_val = preprocess_input(X_val)

In [66]:
class RocCallback(Callback):
    def __init__(self,x_v, y_v):
        self.x_val = x_v
        self.y_val = y_v
    def on_train_begin(self, logs={}):
        return

    def on_train_end(self, logs={}):
        return

    def on_epoch_begin(self, epoch, logs={}):
        return

    def on_epoch_end(self, epoch, logs={}):
        y_pred_val = self.model.predict(self.x_val, verbose = 1)
        roc_val = roc_auc_score(self.y_val, y_pred_val, average = 'macro')
        print('\rval_roc-auc: %s' % (str(round(roc_val,4))),end=10*' ')
        return

    def on_batch_begin(self, batch, logs={}):
        return

    def on_batch_end(self, batch, logs={}):
        return
    
roc = RocCallback(X_val, y_val)

In [67]:
callbacks_list = [
    #roc, 
    
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss', # Use validation loss to monitor the model
        verbose=1,
        patience = 3 # Stop after 3 epochs with no improvements in validation loss
    ),
    
    tf.keras.callbacks.ModelCheckpoint(
        filepath = 'new.hdf5', # file where the checkpoint is saved
        monitor = 'val_loss', # Don't overwrite the saved model unless val_loss is worse
        verbose=1,
        save_best_only = True # Only save model if it is the best
    )]


In [68]:
#https://www.programmersought.com/article/60001511310/
def focal_loss(alpha = 0.5, beta = 2.0):

    epsilon = 1.e-7
    def loss_fn2(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
        alpha_t = y_true*alpha + (tf.ones_like(y_true)-y_true)*(1-alpha)
        y_t = tf.multiply(y_true, y_pred) + tf.multiply(1-y_true, 1-y_pred)
        ce = -tf.log(y_t)
        weight = tf.pow(tf.subtract(1., y_t), beta)
        fl = tf.multiply(tf.multiply(weight, ce), alpha_t)
        loss = tf.reduce_mean(fl)
        return loss
    
    return loss_fn2

In [69]:
D_model = load_model(modelname, custom_objects={ 'loss_fn2': focal_loss()})    #model has been trained with augmentation

'''
model_json_path = 'D.json' #just original model

with open(model_json_path) as infile:
    json_model = json.load(infile)

D_model = keras.models.model_from_json(json.dumps(json_model), custom_objects = {'GlorotUniform': glorot_uniform()})

weights_path = 'D_weights.h5'
D_model.load_weights(weights_path)

D_model.compile(Adam(lr=0.0001), loss = focal_loss())
#'''

"\nmodel_json_path = '1D.json' #'VGG16_112.json' #\n\nwith open(model_json_path) as infile:\n    json_model = json.load(infile)\n\nD_model = keras.models.model_from_json(json.dumps(json_model), custom_objects = {'GlorotUniform': glorot_uniform()})\n\nweights_path = '1D_weights.h5'#'VGG16_112_weights.h5' #\nD_model.load_weights(weights_path)\n\n#D_model.compile(Adam(lr=0.0001), loss = custom_loss())\nD_model.compile(Adam(lr=0.0001), loss = focal_loss())\n#"

In [70]:
nr.seed(1234)
tf.set_random_seed(4321)
history = D_model.fit(X_train, y_train, batch_size=10, 
                          validation_data=(X_val, y_val),
                          epochs=20, verbose=1, 
                          callbacks = callbacks_list)

Train on 22000 samples, validate on 5940 samples
Epoch 1/20
22000/22000 [==============================] - 85s 4ms/step - loss: 0.0327 - val_loss: 0.0571

Epoch 00001: val_loss improved from inf to 0.05713, saving model to new.hdf5
Epoch 2/20
22000/22000 [==============================] - 75s 3ms/step - loss: 0.0322 - val_loss: 0.0557

Epoch 00002: val_loss improved from 0.05713 to 0.05568, saving model to new.hdf5
Epoch 3/20
22000/22000 [==============================] - 76s 3ms/step - loss: 0.0318 - val_loss: 0.0548

Epoch 00003: val_loss improved from 0.05568 to 0.05480, saving model to new.hdf5
Epoch 4/20
22000/22000 [==============================] - 76s 3ms/step - loss: 0.0313 - val_loss: 0.0538

Epoch 00004: val_loss improved from 0.05480 to 0.05384, saving model to new.hdf5
Epoch 5/20
22000/22000 [==============================] - 75s 3ms/step - loss: 0.0312 - val_loss: 0.0495

Epoch 00005: val_loss improved from 0.05384 to 0.04950, saving model to new.hdf5
Epoch 6/20
22000/220

# 6. Training Performance

In [71]:
D_model = load_model('new.hdf5', custom_objects={ 'loss_fn2': focal_loss()})

# 7. Predictive Performance

In [72]:
test_loss = D_model.evaluate(X_test, y_test, batch_size = 8, verbose=1)

test_pred = D_model.predict(X_test, verbose=1)
test_auc_score = roc_auc_score(y_test, test_pred, average = 'macro')

print("Test Loss: {:.4f}".format(test_loss))
print("Test AUC: {:.4f}".format(test_auc_score))

12060/12060 [==============================] - 20s 2ms/step
Test Loss: 0.0499
Test AUC: 0.6270


In [ ]:
plt.figure(1)
plt.plot([0, 1], [0, 1], 'k--')

for i in range(len(disease_class.keys())):
    fpr, tpr, threshold = roc_curve(np.ravel(y_test[:, i]), np.ravel(test_pred[:, i]))
    auc_score = roc_auc_score(y_test[:, i], test_pred[:,i])
    
    plt.plot(fpr, tpr, label=disease_rev[i+1]+' (area = {:.4f})'.format(auc_score))

plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve : VGG16 (test)')
plt.legend(loc='lower left', bbox_to_anchor=(0, 1.07, 1, 0.2))
plt.show()